In [ ]:
# reading the output of zeroed-out interaction inference

import uproot
import os
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

In [ ]:
with uproot.open("../JetClass_zeroed_interaction_inference_HToBB.root") as file:
    tree = file["Events"]
    arrays = tree.arrays(library='np')
    print(arrays.keys())

In [ ]:
# overall accuracy, TPR, FPR
num_events = len(arrays['_label_'])
num_correct = np.sum(arrays['_label_'] == arrays['predicted_label'])
accuracy = num_correct / num_events


In [ ]:
print(arrays['_label_'])
#print(arrays['predicted_label'])
print(arrays['label_Hbb'])
print(arrays['score_label_Hbb'])

In [ ]:
scores = np.stack([arrays[i] for i in arrays.keys() if i.startswith('score_')], axis=1)
#print(scores.shape)
#print(scores[0,:])

# check that they sum to 1
#print(np.sum(scores, axis=1))

y_pred = np.argmax(scores, axis=1)
y_true = arrays['_label_']

#print(y_pred[:10])
acc = np.sum(y_pred == y_true) / len(y_true)
#print(f"Accuracy from scores: {acc}")


In [ ]:
# Getting the ROC curve for Hbb vs non-Hbb
from sklearn.metrics import confusion_matrix, roc_curve, auc
cnf_matrix = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cnf_matrix)

In [ ]:
def rejection(fpr, target_tpr, scores, labels):
    # Find the threshold that gives the target TPR
    idx = np.argmax(tpr >= target_tpr)
    threshold = thresholds[idx]
    # Calculate the rejection (1 / FPR) at this threshold
    fpr_at_threshold = fpr[idx]
    if fpr_at_threshold == 0:
        return float('inf')  # Infinite rejection if FPR is 0
    return 1.0 / fpr_at_threshold